# Data Aggregation & Grouping Insights

### 1. Summary Statistics
* **`median()` vs `mean()`**: Always prefer `median()` for financial data (like Salary) to avoid skewness caused by extreme outliers.
* **`describe()`**: A powerful tool for a quick statistical overview of all numerical features in the dataset.

### 2. The GroupBy Mechanics (`Split-Apply-Combine`)
* **Grouping**: `df.groupby('Column')` does not compute anything immediately; it creates a `GroupBy object` pointing to memory segments.
* **`get_group()`**: Useful for inspecting all data related to a single category (e.g., all respondents from 'Japan').
* **Aggregating**: Use `.agg(['func1', 'func2'])` to perform multiple statistical operations simultaneously on a group.

### 3. Advanced Filtering within Groups
* **The `.str` Limitation**: Since GroupBy objects don't support direct `.str` access, we use `.apply(lambda x: x.str.contains(...))` to process string data within each group.

### 4. Data Synthesis (The Python Percentage Task)
* **Combining Results**: Using `pd.concat` with `axis='columns'` allows us to merge independent Series (Total Respondents vs. Python Users) into a single analytical DataFrame.
* **Feature Calculation**: Creating a percentage column `PctKnowsPython` is a classic example of creating an "Engineered Feature" for comparative analysis.
### 5. Data Alignment in Concatenation
* **Index Alignment**: When concatenating multiple Series with the same Index (e.g., Country names), Pandas automatically aligns the rows based on the index labels, regardless of their original order.
* **The `sort=False` Parameter**: This prevents Pandas from sorting non-aligned columns alphabetically, which optimizes performance and preserves the original column order.
### 6.  The Importance of the `axis` Parameter
* **`axis=0` (Default)**: Concatenates objects vertically (stacks rows). This would result in a single long column.
* **`axis=1` or `'columns'`: Concatenates objects horizontally (aligns columns). This is essential when merging different metrics (like counts and sums) for the same index (Country).
* **Engineering Note**: Missing the `axis` parameter is a common source of logic errors in data pipelines. Always specify it when building feature tables.

In [ ]:
import pandas as pd

df = pd.read_csv(r"data\survey_results_public.csv")
df_schema = pd.read_csv(r"data\survey_results_schema.csv", index_col="Column")
pd.set_option('display.max_columns', 85)
pd.set_option('display.max_rows', 85)

In [ ]:
df['ConvertedComp'].head(15)

In [ ]:
# __ median() __
df['ConvertedComp'].median()  # here median() return the average of items in the column "ConvertedComp"

In [ ]:
df.median(numeric_only=True)  # here median() return the average of items in the all columns in dataframe
# but you need to put numeric_only=True because dataframe may contain strings

In [ ]:
# describe() => return many value from dataframe like [count,mean,std,min,25%,50%,75%,max] only numeric columns
df.describe()

In [ ]:
df['ConvertedComp'].count()  # return non-Empty cell

In [ ]:
df['Hobbyist'].value_counts()  # here return series of yes_count and no_count


In [ ]:
df["SocialMedia"]

In [ ]:
df_schema.loc['SocialMedia', "QuestionText"]
# df_schema.loc['SocialMedia']

In [ ]:
df['SocialMedia'].value_counts()  # here return series of Reddit_count , YouTube _count , ......

In [ ]:
df['SocialMedia'].value_counts(
    normalize=True)  # here return series of Reddit_count , YouTube _count , ...... but in percentage of 1

In [ ]:
df['SocialMedia'].value_counts(
    normalize=True) * 100  # here return series of Reddit_count , YouTube _count , ...... but in percentage of 100

In [ ]:
df['Country'].value_counts()

In [ ]:
Country_grp = df.groupby('Country')
Country_grp.get_group('United States')
# ok here for main dataframe defined a Country_grp = <pandas.api.typing.DataFrameGroupBy object at 0x0000021B687C6E90>
# and when use this object.get_group(cell from column)
# return the df grouped by cell in a column

In [ ]:
df['Country'].unique()  # return a <StringArray> of cells

In [ ]:
mask = df['Country'] == "United States"

In [ ]:
df.loc[mask]["SocialMedia"].value_counts()  # ok here return a series of SocialMedia platform in United States (mask)

In [ ]:
# Country_grp = df.groupby('Country')
Country_grp["SocialMedia"].value_counts().loc["India"].head(10)
# ok here return a series of SocialMedia platform in "India" but the different between mask and
# .groupby is every time i loc for a different Country in mask I should redefine a new mask but in  .groupby just
# replace the cell

In [ ]:
Country_grp["SocialMedia"].value_counts().loc["United States"].head(10)


In [ ]:
Country_grp["SocialMedia"].value_counts().loc["China"].head(10)

In [ ]:
(Country_grp["SocialMedia"].value_counts(normalize=True) * 100).loc["Russian Federation"].head(10)
# return a percentage of 100 in "Russian Federation" ..........

In [ ]:
Country_grp["ConvertedComp"].median()
Country_grp["ConvertedComp"].max()
Country_grp["ConvertedComp"].min()
# here return the series of Country how  ConvertedComp is min or max or ..... like the min salary in Afghanistan is 0.0 and in Albania is 1320.0

In [ ]:
Country_grp["ConvertedComp"].median().loc['Afghanistan']
# return specific value  of the average salary in Afghanistan

In [ ]:
Country_grp["ConvertedComp"].agg(["median", "mean"])
# ok here return dataframe of Countries to which specific functions such as arithmatic "median" "mean" were applied are returned

In [ ]:
Country_grp["ConvertedComp"].agg(["median", "mean"]).loc['Canada']
#  here return series of Canada to which specific functions such as arithmatic "median" "mean" were applied are returned

In [ ]:
mask = df['Country'] == "United States"
df.loc[mask]['LanguageWorkedWith'].str.contains("Python").sum()
# here return a value_count of people in "United States" code by Python

In [ ]:
# Country_grp["LanguageWorkedWith"].str.contains("Python").sum()
# error AttributeError: 'SeriesGroupBy' object has no attribute 'str'
# to solve that def a func
Country_grp["Langua geWorkedWith"].apply(lambda x: x.str.contains("Python").sum())
# from Afghanistan 8 people know python .....

In [ ]:
Country_respondents = df["Country"].value_counts()

In [ ]:
Country_respondents

In [ ]:
Country_Uses_Python = Country_grp["LanguageWorkedWith"].apply(lambda x: x.str.contains("Python").sum())

In [ ]:
Country_Uses_Python

In [ ]:
python_df = pd.concat([Country_respondents, Country_Uses_Python], axis='columns', sort=False)

In [ ]:
python_df

In [ ]:
python_df.rename(columns={'count': "NumRespondents", "LanguageWorkedWith": "NumPeople_Know_Python"}, inplace=True)

In [ ]:
python_df["PctKnowsPython"] = (python_df["NumPeople_Know_Python"] / python_df["NumRespondents"]) * 100

In [ ]:
python_df

In [ ]:
python_df.sort_values(by="PctKnowsPython", ascending=False, inplace=True)

In [ ]:
python_df.head(50)

In [ ]:
python_df.loc["Japan"]

In [ ]:
# python_df.fillna(0, inplace=True)